# {class}`~tvm.contrib.msc.pipeline.manager.MSCManager`

{class}`~tvm.contrib.msc.pipeline.manager.MSCManager` 将 MSCGraph(s) 与不同的框架连接起来，它封装了一些常用的方法并管理 MSCTools。

In [1]:
%cd ..
import set_env
from pathlib import Path

temp_dir = Path(".temp")
temp_dir.mkdir(exist_ok=True)

/media/pc/data/lxw/ai/tvm-book/doc/tutorials/msc


构建前端模型：

In [2]:
import numpy as np
import torch
from torch import fx
import tvm
from tvm import relax
from tvm.relax.frontend.torch import from_fx

class M(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = torch.nn.Conv2d(3, 6, 1, bias=False)
        self.relu = torch.nn.ReLU()

    def forward(self, data):
        x = self.conv(data)
        return self.relu(x)

input_info = [([1, 3, 224, 224], "float32")]
with torch.no_grad():
    torch_fx_model = fx.symbolic_trace(M())
    mod = from_fx(torch_fx_model, input_info, keep_params_as_input=False)
mod, params = relax.frontend.detach_params(mod)

```python
from tvm.contrib.msc.core.transform import msc_transform
from tvm.contrib.msc.core.runtime import create_runtime_manager
from tvm.contrib.msc.core.tools import create_tool, MSC_TOOL

# build runtime manager from module and mscgraphs
optimized_mod, msc_graph, msc_config = msc_transform(mod, params)
rt_manager = create_runtime_manager(optimized_mod, params, msc_config)
rt_manager.create_tool(MSC_TOOL.QUANTIZE, quantize_config)
quantizer = rt_manager.get_tool(MSC_TOOL.QUANTIZE)

rt_manager.load_model()
# calibrate the datas with float model
while not quantizer.calibrated:
    for datas in calibrate_datas:
        rt_manager.run(datas)
    quantizer.calibrate()
quantizer.save_strategy(strategy_file)

# load again the quantized model, without loading the weights
rt_manager.load_model(reuse_weights=True)
outputs = rt_manager.run(sample_datas)
```

MSCManager将编译流程进行封装，暴露出一个面向用户的接口。使用方式类似：
```python
improt torchvision
from tvm.contrib.msc.pipeline import MSCManager

model = trochvision.models.resnet50()
# define your config
manager = MSCManager(model, config)
runner = manager.run_pipe()
```

## MSCWrapper

MSCWrapper是对MSCManger的进一步封装，主要作用是将MSC的编译过程变成用户友好的工具连接口。其使用方式和MSCManager基本相同，如
```python
model = TorchWrapper(model, config)

# export to dump meta model
# model.export()

# optimize the model with quantizer(PTQ)
model.optimize()
acc = eval_model(model, testloader, max_iter=args.test_iter)

# train the model with quantizer(QAT)
optimizer = optim.Adam(model.parameters(), lr=0.0000001, weight_decay=0.08)
for ep in range(args.train_epoch):
    train_model(model, trainloader, optimizer, max_iter=args.train_iter)
    acc = eval_model(model, testloader, max_iter=args.test_iter)

# export to dump checkpoint model
# model.export()

# compile the model
model.compile(bind_params=True)
acc = eval_model(model, testloader, max_iter=args.test_iter)

# export to dump compiled model
# model.export()
```

使用example尚未合入，合入后更新文档。

MSCWrapper包裹的model保留原model所有的方法，可以用于训练或者评测过程，但调用MSCWrapper.optimize或MSCWrapper.compile之后model已经被替换成了优化之后或编译得到的模型，只在输入输出格式上进行适配支持原始模型对应格式的数据类型。